In [229]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor


## **Load Data**

In [230]:
print(f"Current working directory: {os.getcwd()}")
os.chdir("/Users/derekwu/Desktop/Data/01_silver")
print(f"Current working directory: {os.getcwd()}")

data = pd.read_csv("clean_chs_waittimes.csv")

Current working directory: /Users/derekwu/Desktop/Data/01_silver
Current working directory: /Users/derekwu/Desktop/Data/01_silver


## **Encoding**

In [231]:
from sklearn.preprocessing import LabelEncoder

# Encode 'id' and 'clinic_type'
label_encoder_id = LabelEncoder()
label_encoder_clinic_type = LabelEncoder()

data['id_encoded'] = label_encoder_id.fit_transform(data['id'])
data['clinic_type_encoded'] = label_encoder_clinic_type.fit_transform(data['clinic_type'])

# Drop original categorical columns to avoid redundancy
data = data.drop(columns=['id', 'clinic_type'])

# Map days of the week to numeric values
day_mapping = {
    'Monday': 0, 'Tuesday': 1, 'Wednesday': 2,
    'Thursday': 3, 'Friday': 4, 'Saturday': 5, 'Sunday': 6
}

data['day_of_week_encoded'] = data['day_of_week'].map(day_mapping)

data = data.dropna()

In [232]:
for col in data.columns:
    print(col)

lastUpdated
queue
wait_time
treat_time
total_time
hour_minute
hour_minute_numeric
day_of_week
hour_time
date
queue_diff
wait_time_diff
treat_time_diff
total_time_diff
hour_minute_numeric_diff
lagged_hour_time
lagged_hour_minute_numeric
lagged_hour_minute_numeric_diff
lagged_wait_time
lagged_treat_time
lagged_total_time
lagged_queue
lagged_queue_diff
id_encoded
clinic_type_encoded
day_of_week_encoded


## **Model**

In [233]:
set(data['clinic_type_encoded'])

{0, 1}

In [234]:
import numpy as np

def ts_train_test_split(X, y, group_col, time_col, test_size=0.2, random_state=42):
    """
    Custom train-test split that splits data by group and time, ensuring reproducibility.

    Parameters:
        X (pd.DataFrame): Feature dataset to split.
        y (pd.Series or pd.DataFrame): Target variable to split.
        group_col (str): The column used to group the data.
        time_col (str): The column used to order the data within each group.
        test_size (float): The proportion of data to use for testing (default is 0.2).
        random_state (int): Random seed for reproducibility.

    Returns:
        X_train, X_test, y_train, y_test: Split datasets.
    """
    # Combine X and y for consistent indexing
    data = X.copy()
    data['target'] = y

    np.random.seed(random_state)  # Set random seed for reproducibility
    
    train_indices = []
    test_indices = []

    # Group by the group_col
    for group in data[group_col].unique():
        group_data = data[data[group_col] == group].sort_values(by=time_col)
        n_samples = len(group_data)
        n_test = int(n_samples * test_size)

        # Shuffle indices deterministically
        shuffled_indices = group_data.index.to_numpy()
        np.random.shuffle(shuffled_indices)

        # Split shuffled indices into train and test sets
        train_idx = shuffled_indices[:-n_test]  # All but the last `n_test` samples
        test_idx = shuffled_indices[-n_test:]  # The last `n_test` samples

        train_indices.extend(train_idx)
        test_indices.extend(test_idx)

    # Split into train and test sets
    train_data = data.loc[train_indices]
    test_data = data.loc[test_indices]

    return (
        train_data.drop(columns=['target']),
        test_data.drop(columns=['target']),
        train_data['target'],
        test_data['target']
    )

In [235]:
data[data['clinic_type_encoded'] == 0]

,lastUpdated,queue,wait_time,treat_time,total_time,hour_minute,hour_minute_numeric,day_of_week,hour_time,date,...,lagged_hour_minute_numeric,lagged_hour_minute_numeric_diff,lagged_wait_time,lagged_treat_time,lagged_total_time,lagged_queue,lagged_queue_diff,id_encoded,clinic_type_encoded,day_of_week_encoded
27548,2024-10-29 22:18:00.000,24,3.150000,2.516667,5.666667,1900-01-01 22:18:00,1338,Tuesday,22:18:00,2024-10-29,...,1275.0,0.0,3.100000,3.150000,3.150000,24.0,0.0,1,0,1
27549,2024-10-29 22:21:00.000,23,3.100000,2.533333,5.633333,1900-01-01 22:21:00,1341,Tuesday,22:21:00,2024-10-29,...,1278.0,3.0,3.150000,3.200000,3.200000,25.0,1.0,1,0,1
27550,2024-10-29 22:21:00.000,23,3.100000,2.533333,5.633333,1900-01-01 22:21:00,1341,Tuesday,22:21:00,2024-10-29,...,1278.0,0.0,3.150000,3.200000,3.200000,25.0,0.0,1,0,1
27551,2024-10-29 22:21:00.000,23,3.100000,2.533333,5.633333,1900-01-01 22:21:00,1341,Tuesday,22:21:00,2024-10-29,...,1278.0,0.0,3.150000,3.200000,3.200000,25.0,0.0,1,0,1
27552,2024-10-29 22:24:00.000,24,3.150000,2.483333,5.633333,1900-01-01 22:24:00,1344,Tuesday,22:24:00,2024-10-29,...,1281.0,3.0,3.150000,3.216667,3.216667,23.0,-2.0,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
181312,2024-12-10 21:45:00.000,12,2.683333,3.716667,6.400000,1900-01-01 21:45:00,1305,Tuesday,21:45:00,2024-12-10,...,1221.0,1.0,2.833333,3.833333,3.833333,8.0,0.0,4,0,1
181313,2024-12-10 21:46:00.000,12,2.683333,3.716667,6.400000,1900-01-01 21:46:00,1306,Tuesday,21:46:00,2024-12-10,...,1222.0,1.0,2.766667,3.666667,3.666667,7.0,-1.0,4,0,1
181314,2024-12-10 21:47:00.000,12,2.683333,3.716667,6.400000,1900-01-01 21:47:00,1307,Tuesday,21:47:00,2024-12-10,...,1223.0,1.0,2.766667,3.816667,3.816667,7.0,0.0,4,0,1
181315,2024-12-10 21:48:00.000,13,2.766667,3.716667,6.483333,1900-01-01 21:48:00,1308,Tuesday,21:48:00,2024-12-10,...,1225.0,2.0,2.766667,3.816667,3.816667,7.0,0.0,4,0,1


In [236]:
X = data[data['clinic_type_encoded'] == 0][['queue', 'treat_time', 'queue_diff', 'hour_minute_numeric', 'hour_minute_numeric_diff', 'id_encoded', 'clinic_type_encoded', 'day_of_week_encoded',
                                            'lagged_queue', 'lagged_treat_time', 'lagged_queue_diff', 'lagged_hour_minute_numeric', 'lagged_hour_minute_numeric_diff']]
y = data[data['clinic_type_encoded'] == 0][['wait_time']]

In [245]:
# X.sort_values(by = ['date', 'hour_time'], ascending = True)

In [246]:
# Split data into training and testing sets
# WIC == 1, ED == 0

# X = data[data['clinic_type_encoded'] == 0][['queue', 'treat_time', 'queue_diff', 'hour_minute_numeric', 'hour_minute_numeric_diff', 'id_encoded', 'clinic_type_encoded', 'day_of_week_encoded',
#                                             'lagged_queue', 'lagged_treat_time', 'lagged_queue_diff', 'lagged_hour_minute_numeric', 'lagged_hour_minute_numeric_diff']]
# y = data[data['clinic_type_encoded'] == 0][['wait_time']]

X = data[data['clinic_type_encoded'] == 1][['queue', 'treat_time', 'queue_diff', 'hour_minute_numeric', 'hour_minute_numeric_diff', 'id_encoded', 'clinic_type_encoded', 'day_of_week_encoded',
                                           'lagged_queue', 'lagged_treat_time', 'lagged_queue_diff', 'lagged_hour_minute_numeric', 'lagged_hour_minute_numeric_diff']]
y = data[data['clinic_type_encoded'] == 1][['wait_time']]

# Align indices
X = X.reset_index(drop=True)
y = y.reset_index(drop=True)


X_train, X_test, y_train, y_test = ts_train_test_split(X, y, group_col='clinic_type_encoded', time_col='hour_minute_numeric', test_size=0.1, random_state=42)

In [247]:
def calculate_mape(y_true, y_pred):
    """
    Calculates Mean Absolute Percentage Error (MAPE).
    """
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

def calculate_smape(y_true, y_pred):
    """
    Calculates Symmetric Mean Absolute Percentage Error (SMAPE).
    """
    return 100 * np.mean(np.abs(y_true - y_pred) / ((np.abs(y_true) + np.abs(y_pred)) / 2))

In [248]:
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error
import itertools
import numpy as np

def evaluate_arima_with_optimization(y_train, y_test, p_range, d_range, q_range):
    """
    Optimizes ARIMA parameters and evaluates the best model.
    """
    best_order = None
    best_mse = float('inf')
    best_rmse = float('inf')
    best_mape = float('inf')
    best_smape = float('inf')
    best_model = None

    # Generate all combinations of p, d, q
    pdq_combinations = list(itertools.product(p_range, d_range, q_range))

    for order in pdq_combinations:
        try:
            # Fit ARIMA model
            model = ARIMA(y_train, order=order)
            model_fit = model.fit()
            
            # Forecast on the test set
            y_pred = model_fit.forecast(steps=len(y_test))
            
            # Calculate Metrics
            mse = mean_squared_error(y_test, y_pred)
            rmse = np.sqrt(mse)
            mape = calculate_mape(y_test, y_pred)
            smape = calculate_smape(y_test, y_pred)

            # Update best model
            if mse < best_mse:
                best_mse, best_rmse, best_mape, best_smape = mse, rmse, mape, smape
                best_order = order
                best_model = model_fit
        except Exception as e:
            print(f"ARIMA fitting failed for order {order}: {e}")
            continue
    
    # Handle cases where no valid model was found
    if best_model is None:
        return None, None, float('nan'), float('nan'), float('nan'), float('nan')
    
    return best_model, best_order, best_mse, best_rmse, best_mape, best_smape

In [249]:
# Define the custom grouped time-series splitter
class GroupedTimeSeriesSplit:
    def __init__(self, group_col, time_col, n_splits):
        self.group_col = group_col
        self.time_col = time_col
        self.n_splits = n_splits

    def split(self, X, y=None):
        """Yield train-test splits for each group."""
        groups = X[self.group_col].unique()
        for group in groups:
            group_data = X[X[self.group_col] == group].sort_values(self.time_col)
            n_samples = len(group_data)
            
            # Skip small groups
            if n_samples <= self.n_splits:
                print(f"Skipping group '{group}' due to insufficient samples.")
                continue
            
            fold_size = n_samples // (self.n_splits + 1)
            if fold_size == 0:
                print(f"Skipping group '{group}' due to insufficient samples per fold.")
                continue

            for i in range(1, self.n_splits + 1):
                start_train = 0
                end_train = fold_size * i
                start_test = fold_size * i
                end_test = fold_size * (i + 1)

                # Ensure indices are within bounds
                if end_train > n_samples or end_test > n_samples:
                    print(f"Skipping fold {i} for group '{group}' due to index out-of-bounds.")
                    continue

                train_idx = group_data.index[start_train:end_train]
                test_idx = group_data.index[start_test:end_test]

                # Ensure indices are not empty
                if len(train_idx) == 0 or len(test_idx) == 0:
                    print(f"Skipping fold {i} for group '{group}' due to empty train/test indices.")
                    continue

                yield train_idx, test_idx

# Define a function for model evaluation
def evaluate_model_with_grouped_cv(model, X, y, splitter, p_range, d_range, q_range):
    mse_scores, rmse_scores, mape_scores, smape_scores, r2_scores = [], [], [], [], []
    
    for train_idx, test_idx in splitter.split(X):
        try:
            # Extract train-test splits
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx].values.ravel(), y.iloc[test_idx].values.ravel()
            
            # Skip if training data is too small
            if len(y_train) < 10:
                print("Skipping split due to insufficient training data.")
                continue
            
            if model == 'ARIMA':
                # Evaluate ARIMA model
                best_model, best_order, mse, rmse, mape, smape = evaluate_arima_with_optimization(
                    y_train, y_test, p_range, d_range, q_range
                )
            else:
                # Evaluate other models
                model.fit(X_train, y_train)
                y_pred = model.predict(X_test)
                
                # # Debug prediction shapes
                # print("y_pred shape:", y_pred.shape)
                # print("y_test shape:", y_test.shape)
                
                mse = mean_squared_error(y_test, y_pred)
                rmse = np.sqrt(mse)
                mape = calculate_mape(y_test, y_pred)
                smape = calculate_smape(y_test, y_pred)
                r2 = r2_score(y_test, y_pred)

            # Append metrics
            mse_scores.append(mse)
            rmse_scores.append(rmse)
            mape_scores.append(mape)
            smape_scores.append(smape)
            r2_scores.append(r2)

        except Exception as e:
            print(f"Error evaluating model: {e}")
            continue
    
    # Return average metrics across folds
    return (
        np.nanmean(mse_scores),
        np.nanmean(rmse_scores),
        np.nanmean(mape_scores),
        np.nanmean(smape_scores),
        np.nanmean(r2_scores),
    )

In [250]:
# Define ARIMA parameter ranges
p_range = range(0, 3)  # AR terms
d_range = range(0, 2)  # Differencing terms
q_range = range(0, 3)  # MA terms

# Define models
models = pd.DataFrame({
    'model_name': ['Linear Regression', 'Random Forest', 'XGBoost', 'ARIMA'],
    'model': [
        LinearRegression(),
        RandomForestRegressor(random_state=42),
        XGBRegressor(random_state=42),
        'ARIMA'  # Placeholder for ARIMA
    ]
})

# Instantiate the custom splitter
custom_splitter = GroupedTimeSeriesSplit(
    group_col='clinic_type_encoded',
    time_col='hour_minute_numeric',
    n_splits=6
)

In [251]:
#### WIC
#### WICS need to be grouped by location i reckon

# Apply the evaluation function
# WIC areas we get about a estimate 14 minute +- error
models[['mse', 'rmse', 'mape', 'smape', 'r2']] = models['model'].apply(
    lambda m: pd.Series(evaluate_model_with_grouped_cv(m, X, y, custom_splitter, p_range, d_range, q_range))
)

# Display results
print(models)

/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Lik

Error evaluating model: cannot access local variable 'r2' where it is not associated with a value


/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Lik

Error evaluating model: cannot access local variable 'r2' where it is not associated with a value


/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregr

...

/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregr

Error evaluating model: cannot access local variable 'r2' where it is not associated with a value


/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregr

Error evaluating model: cannot access local variable 'r2' where it is not associated with a value
          model_name                                              model  \
0  Linear Regression                                 LinearRegression()   
1      Random Forest  (DecisionTreeRegressor(max_features=1.0, rando...   
2            XGBoost  XGBRegressor(base_score=None, booster=None, ca...   
3              ARIMA                                              ARIMA   

        mse      rmse       mape      smape        r2  
0  0.146272  0.379942  51.307392  34.559995  0.298076  
1  0.128094  0.357205  36.054630  29.900723  0.385354  
2  0.123232  0.350489  34.782070  30.316350  0.406107  
3  0.216802  0.464926  61.676529  41.316209       NaN  


/var/folders/my/1zd7hfx53qq246r8xfrrzkdc0000gn/T/ipykernel_64418/1780320840.py:98: RuntimeWarning: Mean of empty slice
  np.nanmean(r2_scores),


In [244]:
#### ED

# Apply the evaluation function
# ED areas we get about a estimate 14 minute +- error
models[['mse', 'rmse', 'mape', 'smape', 'r2']] = models['model'].apply(
    lambda m: pd.Series(evaluate_model_with_grouped_cv(m, X, y, custom_splitter, p_range, d_range, q_range))
)

# Display results
print(models)

/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA param

Error evaluating model: cannot access local variable 'r2' where it is not associated with a value


/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregr

Error evaluating model: cannot access local variable 'r2' where it is not associated with a value


/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregr

...

/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Error evaluating model: cannot access local variable 'r2' where it is not associated with a value


/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Error evaluating model: cannot access local variable 'r2' where it is not associated with a value
          model_name                                              model  \
0  Linear Regression                                 LinearRegression()   
1      Random Forest  (DecisionTreeRegressor(max_features=1.0, rando...   
2            XGBoost  XGBRegressor(base_score=None, booster=None, ca...   
3              ARIMA                                              ARIMA   

        mse      rmse       mape      smape        r2  
0  0.478467  0.652202  23.488506  23.060526  0.114018  
1  0.205123  0.447846  16.591503  16.035377  0.545931  
2  0.209025  0.454806  17.239907  16.629018  0.524323  
3  0.562960  0.735291  30.035678  26.224055       NaN  


/var/folders/my/1zd7hfx53qq246r8xfrrzkdc0000gn/T/ipykernel_64418/1780320840.py:98: RuntimeWarning: Mean of empty slice
  np.nanmean(r2_scores),
